In [31]:
from games.leduc import Leduc
from agents.counterfactualregret_t import CounterFactualRegret
from agents.agent_random import RandomAgent
from agents.ismcts import ISMCTS
import numpy as np
from collections import defaultdict
from tqdm import tqdm

In [32]:
def mapActionName(action):
    return ['call', 'raise', 'fold', 'check'][action]

In [33]:
game = Leduc(render_mode="human")
game.reset(seed=4)

print("Agents:", game.agents)
print("Current agent:", game.agent_selection)

print("Observation player_0", game.observe(game.agents[0]))
print("Observation player_1", game.observe(game.agents[1]))

print("Legal actions:", [mapActionName(action) for action in game.available_actions()])

Agents: ['player_0', 'player_1']
Current agent: player_1
Observation player_0 K_#_2_1_1
Observation player_1 J_#_2_1_1
Legal actions: ['call', 'raise', 'fold']


In [25]:
agents_random = {
    agent: RandomAgent(game=game, agent=agent)
    for agent in game.agents
}
agents_random

{'player_0': <agents.agent_random.RandomAgent at 0x276a8d46d50>,
 'player_1': <agents.agent_random.RandomAgent at 0x276a9883a90>}

In [26]:
game.reset(seed=1)

while not game.game_over():
    agent = game.agent_selection
    legal_actions = game.available_actions()
    action = agents_random[agent].action()

    game.render()
    print(f"Agent {agent}")
    print(f"Observation: {game.observe(agent)}")
    print(f"Legal actions: {[mapActionName(action) for action in legal_actions]}")
    print(f"Action '{mapActionName(action)}'")
    print()

    game.step(action)

game.render()
for agent in game.agents:
    print(f"Reward {agent} = {game.reward(agent)}")

Agent player_0
Observation: K_#_1_2_0
Legal actions: ['call', 'raise', 'fold']
Action 'fold'

Reward player_0 = -0.5
Reward player_1 = 0.5


In [ ]:
game = Leduc(render_mode="")
game.reset(seed=1)

agents = [
    CounterFactualRegret(game=game, agent=game.agents[0]),
    ISMCTS(game=game, agent=game.agents[1], simulations=200, rollouts=10, rollout_iterations=100)
    #CounterFactualRegret(game=game, agent=game.agents[1])
]
agents

In [ ]:
n_training_iterations = 100

for i, agent in enumerate(game.agents):
    print(f"Training agent {agent}")
    if hasattr(agents[agent], 'train'):
        agents[i].train(n_training_iterations)
        print(f"Known information sets: {len(agents_cfr[i].node_dict)}")

Training agent player_0


100%|██████████| 100/100 [00:33<00:00,  2.99it/s]


Known information sets: 576
Training agent player_1


100%|██████████| 100/100 [00:31<00:00,  3.19it/s]

Known information sets: 576


In [29]:
for i, agent in enumerate(game.agents):
    print(f"First 10 policies for {agent} (total information sets: {len(agents_cfr[i].node_dict)})")
    for obs, node in list(agents_cfr[i].node_dict.items())[:10]:
        policy = [(mapActionName(move), round(float(prob), 3)) for move, prob in enumerate(node.policy(node.game.available_actions()))]
        print(obs, policy)
    print()

First 10 policies for player_0 (total information sets: 576)
K_#_1_2_0 [('call', 0.209), ('raise', 0.778), ('fold', 0.013), ('check', 0.0)]
Q_#_2_2_0c [('call', 0.0), ('raise', 0.128), ('fold', 0.025), ('check', 0.847)]
K_#_2_4_0cr [('call', 0.434), ('raise', 0.553), ('fold', 0.013), ('check', 0.0)]
Q_K_4_4_0crc [('call', 0.0), ('raise', 0.53), ('fold', 0.029), ('check', 0.442)]
K_K_4_8_0crcr [('call', 0.082), ('raise', 0.886), ('fold', 0.032), ('check', 0.0)]
Q_K_12_8_0crcrr [('call', 0.971), ('raise', 0.0), ('fold', 0.029), ('check', 0.0)]
K_K_4_4_0crck [('call', 0.0), ('raise', 0.879), ('fold', 0.032), ('check', 0.089)]
Q_K_8_4_0crckr [('call', 0.375), ('raise', 0.596), ('fold', 0.029), ('check', 0.0)]
K_K_8_12_0crckrr [('call', 0.967), ('raise', 0.0), ('fold', 0.033), ('check', 0.0)]
Q_#_6_4_0crr [('call', 0.985), ('raise', 0.0), ('fold', 0.015), ('check', 0.0)]

First 10 policies for player_1 (total information sets: 576)
Q_#_2_1_1 [('call', 0.336), ('raise', 0.606), ('fold', 0.05

In [30]:
cum_rewards = defaultdict(float)
n_games = 1

for _ in tqdm(range(n_games)):
    game.reset()
    while not game.game_over():
        agent = game.agent_selection
        agent_index = game.agents.index(agent)
        action = agents_cfr[agent_index].action()
        obs = game.observe(agent)
        print(f"Agent {agent} - Observation: {obs} - Action: {mapActionName(action) if n_games == 1 else None}")
        game.step(action)

    for agent in game.agents:
        cum_rewards[agent] += game.reward(agent)

print("Average rewards:")
for agent in game.agents:
    print(f"{agent}: {cum_rewards[agent] / n_games:.3f}")

100%|██████████| 1/1 [00:00<00:00, 500.57it/s]

Agent player_1 - Observation: K_#_2_1_1 - Action: raise
Agent player_0 - Observation: Q_#_2_4_1r - Action: raise
Agent player_1 - Observation: K_#_6_4_1rr - Action: call
Agent player_0 - Observation: Q_J_6_6_1rrc - Action: check
Agent player_1 - Observation: K_J_6_6_1rrck - Action: raise
Agent player_0 - Observation: Q_J_6_10_1rrckr - Action: raise
Agent player_1 - Observation: K_J_14_10_1rrckrr - Action: call
Average rewards:
player_0: -7.000
player_1: 7.000
